# 04 — Fit Recommendation (rule-based)

Purpose: Compare estimated fit signals with garment measurements (data/garment/garment_measurements.csv) and recommend S / M / L using ease tables and rule-based scoring.

Outputs:
- Recommended size
- Score (closer to 0 means better fit)
- Confidence (visibility × fit closeness)
- Reasons (	oo_tight / 	oo_loose / good_fit)

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.measurement import FitSignals
from src.fit_rules import recommend_size

GARMENT_CSV = ROOT / "data" / "garment" / "garment_measurements.csv"
SIGNALS_CSV = ROOT / "data" / "processed" / "measurements" / "fit_signals.csv"

In [ ]:
garments = pd.read_csv(GARMENT_CSV)
signals_df = pd.read_csv(SIGNALS_CSV)
garments.head(), signals_df.head()

In [ ]:
results = []
for _, row in signals_df.iterrows():
    sig = FitSignals(
        shoulder_width_cm=row["shoulder_width_cm"],
        hip_width_cm=row["hip_width_cm"],
        torso_length_cm=row["torso_length_cm"],
        leg_length_cm=row["leg_length_cm"],
        arm_length_cm=row["arm_length_cm"],
        detected_body_height_px=row["detected_body_height_px"],
        pixel_to_cm=row["pixel_to_cm"],
        confidence=row["confidence"],
        user_height_cm=row["user_height_cm"],
    )
    for gid in garments["garment_id"].unique():
        rec = recommend_size(garments, sig, preference="regular", garment_id=gid)
        results.append({
            "user_id": row["user_id"],
            "garment_id": gid,
            "size": rec.size,
            "score": round(rec.score, 3),
            "confidence": round(rec.confidence, 3),
            "reasons": "; ".join(rec.reasons),
        })
out = pd.DataFrame(results)
out

In [ ]:
rec_csv = ROOT / "data" / "processed" / "measurements" / "recommendations.csv"
out.to_csv(rec_csv, index=False)
print(f"wrote: {rec_csv}")